# Fondasi Machine Learning

## Bagian 2a: Pengantar Regresi dan Transformasi Variabel Target

### Tujuan Pembelajaran

Setelah materi ini, kita bisa:

- menerapkan transformasi agar variabel target lebih mendekati distribusi normal untuk regresi;
- menerapkan transformasi balik supaya hasil prediksi kembali ke skala aslinya.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import inv_boxcox
from scipy.stats import boxcox, normaltest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")

## Mengapa target perlu normal?

Regresi linear bekerja paling baik ketika residual menyebar normal. Salah satu caranya adalah menstabilkan bentuk distribusi target lebih dulu, lalu memodelkan nilai hasil transformasinya.

Ada dua cara memeriksa kenormalan distribusi:

- **visual**, lewat histogram;
- **statistik**, lewat uji kenormalan.

Uji kenormalan menghasilkan *p-value*. Semakin tinggi nilainya, semakin dekat distribusi ke bentuk normal. Dengan ambang umum 0,05, distribusi dianggap normal bila p-value di atas ambang tersebut.

In [ ]:
rng = np.random.default_rng(72018)


def show_distribution(values, title):
    plt.figure(figsize=(6, 4))
    plt.hist(values, bins=30, edgecolor="white")
    plt.title(title)
    plt.xlabel("value")
    plt.ylabel("frequency")
    plt.show()


def exponential_sample(size=1000):
    return np.exp(rng.normal(size=size))


def squared_normal_sample(loc=5, size=1000):
    return np.square(rng.normal(loc=loc, size=size))

### Memuat data

Data yang dipakai adalah data harga rumah Boston dalam bentuk pickle. Berkas diunduh sekali dari repositori publik, lalu dibaca dengan pandas.

In [ ]:
data_path = Path("data/boston_housing_clean.pickle")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

housing_bundle = pd.read_pickle(data_path)
dataset = housing_bundle["dataframe"]
dataset.head(15)

In [ ]:
target_column = "MEDV"
dataset[target_column].hist(bins=30, edgecolor="white");

Histogram di atas belum simetris: ekornya memanjang ke kanan, tanda distribusi miring positif.

### Uji kenormalan

Fungsi `normaltest` memakai uji D'Agostino K^2. Nilai p yang sangat kecil menandakan hipotesis "distribusi normal" ditolak, sehingga target `MEDV` memang tidak normal.

In [ ]:
normaltest(dataset[target_column].to_numpy())

Hasil uji menegaskan dugaan dari histogram: p-value nyaris nol.

## Transformasi target

Regresi linear mengasumsikan residual berdistribusi normal, dan asumsi itu lebih mudah dipenuhi bila target ditransformasi. Tiga pilihan yang umum:

- transformasi logaritma;
- transformasi akar kuadrat;
- transformasi Box-Cox.

### Transformasi logaritma

Logaritma menarik ekor kanan yang panjang ke arah tengah, sehingga data yang sangat miring menjadi lebih simetris.

In [ ]:
skewed_sample = exponential_sample()
show_distribution(skewed_sample, "Exponential sample")
show_distribution(np.log(skewed_sample), "Log transformed sample")

In [ ]:
log_target = np.log(dataset[target_column])
log_target.hist(bins=30, edgecolor="white");

In [ ]:
normaltest(log_target.to_numpy())

Hasilnya lebih dekat ke bentuk normal dibanding target awal, tetapi belum sepenuhnya normal.

### Transformasi akar kuadrat

Akar kuadrat memberi efek serupa dengan logaritma, tetapi lebih lembut. Pada contoh berikut sisa kemiringan ke kanan masih terlihat tipis.

In [ ]:
square_sample = squared_normal_sample()
show_distribution(square_sample, "Squared normal sample")
show_distribution(np.sqrt(square_sample), "Square root transformed sample")

#### Latihan

Terapkan transformasi akar kuadrat pada target `MEDV`, lalu uji kenormalan hasilnya.

In [ ]:
sqrt_target = np.sqrt(dataset[target_column])
sqrt_target.hist(bins=30, edgecolor="white");
normaltest(sqrt_target.to_numpy())

### Transformasi Box-Cox

Box-Cox adalah transformasi berparameter yang mencari pangkat terbaik agar distribusi sedekat mungkin dengan normal:

$$ \text{boxcox}(y_i) = \frac{y_i^{\lambda} - 1}{\lambda} $$

Anggap saja sebagai perumuman akar kuadrat: akar kuadrat memakai pangkat tetap 0,5, sedangkan Box-Cox membiarkan pangkatnya dicari otomatis.

In [ ]:
boxcox_target, boxcox_lambda = boxcox(dataset[target_column])
boxcox_lambda

In [ ]:
boxcox_target_series = pd.Series(boxcox_target, index=dataset.index)
boxcox_target_series.hist(bins=30, edgecolor="white");
normaltest(boxcox_target)

Dari ketiga transformasi, Box-Cox memberi p-value tertinggi, bahkan bisa melewati 0,05. Target yang sudah normal ini siap dipakai untuk regresi.

## Menguji regresi

Alurnya: siapkan fitur polinomial derajat dua, bagi data 70% latih dan 30% uji, standarkan fitur, transformasi target latih, latih model, lalu prediksi data uji.

In [ ]:
features = dataset.drop(columns=[target_column])
target = dataset[target_column]

polynomial = PolynomialFeatures(degree=2, include_bias=False)
features_poly = polynomial.fit_transform(features)

features_train, features_test, target_train, target_test = train_test_split(
    features_poly, target, test_size=0.3, random_state=72018
)

scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)

#### Latihan

Transformasi yang tepat untuk `target_train` adalah Box-Cox, sama seperti pada target penuh. Simpan nilai lambda-nya karena dibutuhkan saat mengembalikan prediksi ke skala asli.

In [ ]:
train_target_bc, train_lambda = boxcox(target_train)
train_target_bc.shape

In [ ]:
model = LinearRegression()
model.fit(features_train_scaled, train_target_bc)

features_test_scaled = scaler.transform(features_test)
predictions_bc = model.predict(features_test_scaled)

### Transformasi balik

Setiap transformasi punya pasangan balik. Kebalikan dari akar kuadrat adalah pangkat dua, dan Box-Cox bisa dibalik dengan `inv_boxcox` asalkan nilai lambda diketahui.

In [ ]:
restored_target = inv_boxcox(boxcox_target, boxcox_lambda)
np.allclose(restored_target, dataset[target_column].to_numpy())

#### Latihan

1. Kembalikan `predictions_bc` ke skala harga asli.
2. Hitung $R^2$ antara hasil transformasi balik dan `target_test`.

In [ ]:
predictions = inv_boxcox(predictions_bc, train_lambda)
r2_transformed = r2_score(target_test, predictions)
r2_transformed

## Latihan tambahan

Bandingkan dengan model tanpa transformasi: hitung $R^2$ regresi linear langsung pada target asli.

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(features_train_scaled, target_train)
r2_baseline = r2_score(target_test, baseline_model.predict(features_test_scaled))
r2_baseline